### Import Library

In [17]:
import pandas as pd
import re

### Load Dataset

In [ ]:
df = pd.read_csv('../data/dataMentah.csv')
df.head()

,Judul,Waktu,Link,Content,tag1,tag2,tag3,tag4,tag5,source
0,"Viral Isu PHK Buruh Gudang Garam, Said Iqbal: ...",6 September 2025,https://nasional.kompas.com/read/2025/09/06/14...,"JAKARTA, KOMPAS.com – Presiden Konfederasi Se...",Said Iqbal,industri rokok,PT Gudang Garam,PHK massal,phk massal 2025 terbaru,kompas
1,"Gempa M 5,3 Guncang Pulau Doi Maluku Utara","Senin, 12 Agu 2024 21:58 WIB",https://news.detik.com/berita/d-7486691/gempa-...,"Gempa bumi berkekuatan magnitudo (M) 5,3 mengg...",pulau doi,gempa,NaN,NaN,NaN,detik
2,"Toko Emas Palsu di Riau Dibongkar Polisi, Perh...","Rabu, 30 Jul 2025 22:22 WIB",https://news.detik.com/melindungi-tuah-marwah/...,Satreskrim Polres Bengkalis membongkar praktik...,pemalsuan emas,emas palsu,polres bengkalis,polda riau,melindungi tuah marwah,detik
3,Minyakita Tak Sesuai Ukuran juga Ditemukan di ...,"Senin, 10 Mar 2025 23:15 WIB",https://news.detik.com/berita/d-7816829/minyak...,Polisi mendatangi salah satu gudang Minyakita ...,minyakita,kudus,NaN,NaN,NaN,detik
4,"Pimpin LDP, Sanae Takaichi Calon Kuat PM Perem...",4 Oktober 2025 | 14.00 WIB,https://www.tempo.co/internasional/pimpin-ldp-...,"Baca berita dengan sedikit iklan, klik di sin...",jepang,perdana-menteri,sanae-takaichi,perempuan,ldp,tempo


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80472 entries, 0 to 80471
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Judul    80472 non-null  object
 1   Waktu    80472 non-null  object
 2   Link     80472 non-null  object
 3   Content  80463 non-null  object
 4   tag1     78023 non-null  object
 5   tag2     77612 non-null  object
 6   tag3     71602 non-null  object
 7   tag4     56856 non-null  object
 8   tag5     35633 non-null  object
 9   source   80472 non-null  object
dtypes: object(10)
memory usage: 6.1+ MB


In [8]:
df.shape

(80472, 10)

In [4]:
# Cek tag 20 teratas pada tag1
print(df['tag1'].value_counts().head(20))

tag1
Jakarta             865
prabowo subianto    774
kpk                 743
prabowo             635
Prabowo             541
KPK                 501
Pramono Anung       415
Ridwan Kamil        340
kebakaran           339
jokowi              327
israel              311
info-tempo          305
mpr                 260
banjir              258
gempa               253
pramono anung       244
kecelakaan          239
PDI-P               235
Prabowo Subianto    222
mbg                 221
Name: count, dtype: int64


In [5]:
# Cek tag 20 teratas pada tag2
print(df['tag2'].value_counts().head(20))

tag2
jabodetabek         488
prabowo subianto    471
info-tempo          423
prabowo             417
Pilkada Jakarta     352
israel              290
korupsi             290
kpk                 258
KPK                 240
Prabowo             232
Pramono Anung       217
Jokowi              216
pilkada 2024        216
bogor               198
jokowi              195
bmkg                191
imsakiyah           180
pilkada             175
Prabowo Subianto    175
dpr                 167
Name: count, dtype: int64


#### Mapping

In [10]:
def map_label(row):
    # Gabungkan tag1 dan tag2 untuk pengecekan lebih akurat
    tags = f"{str(row['tag1']).lower()} {str(row['tag2']).lower()}"
    
    if any(x in tags for x in ['prabowo', 'jokowi', 'pramono', 'ridwan kamil', 'mpr', 'dpr', 'pdi-p', 'pilkada', 'ldp']):
        return 'Politik'
    elif any(x in tags for x in ['kpk', 'korupsi', 'polisi', 'polres', 'polda', 'hukum', 'sidang']):
        return 'Hukum & Kriminal'
    elif any(x in tags for x in ['kebakaran', 'banjir', 'gempa', 'bmkg', 'kecelakaan', 'bencana']):
        return 'Bencana & Peristiwa'
    elif any(x in tags for x in ['israel', 'internasional', 'luar negeri', 'pm']):
        return 'Internasional'
    elif any(x in tags for x in ['jakarta', 'jabodetabek', 'bogor', 'nasional']):
        return 'Nasional/Regional'
    else:
        return 'Lainnya'

df['category'] = df.apply(map_label, axis=1)

# Cek hasil distribusi kategori baru
print(df['category'].value_counts())

category
Lainnya                53719
Politik                12303
Hukum & Kriminal        5550
Nasional/Regional       4323
Bencana & Peristiwa     3380
Internasional           1197
Name: count, dtype: int64


In [11]:
# Menghapus kategori Lainnya agar model fokus pada kategori yang jelas
df = df[df['category'] != 'Lainnya'].reset_index(drop=True)

# Cek distribusi baru
print(df['category'].value_counts())

category
Politik                12303
Hukum & Kriminal        5550
Nasional/Regional       4323
Bencana & Peristiwa     3380
Internasional           1197
Name: count, dtype: int64


#### Cleaning

In [18]:
def data_cleaning(text):
    # Ubah ke string untuk menghindari error jika ada data non-teks
    text = str(text)

    # 1. Menjadi lowercase di awal agar replace iklan lebih akurat
    text = text.lower()

    # 2. Hapus Iklan (pastikan teks iklan yang dicari juga huruf kecil)
    text = text.replace('baca berita dengan sedikit iklan, klik di sini', '')
    
    # 3. Menghapus URL/Link
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 4. Menghapus tag HTML
    text = re.sub(r'<.*?>', '', text)
    
    # 5. Menghapus tanda baca dan angka (hanya sisakan a-z dan spasi)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 6. Menghapus whitespace berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Cara jalankan ke dataset:
df['cleaned_content'] = df['Content'].apply(data_cleaning)

# Cek hasil 5 teratas
print(df['cleaned_content'].head())

0    gempa bumi berkekuatan magnitudo m mengguncang...
1    presiden prabowo subianto bertemu dengan guber...
2    jakarta kompas com wakil ketua mpr ri eddy soe...
3    tangerang selatan kompas com seorang remaja pe...
4    jakarta kompas com kebakaran yang melanda kawa...
Name: cleaned_content, dtype: object


In [27]:
from IPython.display import display, HTML

# Tampilkan tabel dengan lebar kolom tertentu
display(HTML(df[['cleaned_content']].head(50).to_html().replace('<td>', '<td style="max-width: 2500px; word-wrap: break-word;">')))

,cleaned_content
0,gempa bumi berkekuatan magnitudo m mengguncang kabupaten pulau doi maluku utara gempa berada pada kedalaman km informasi gempa ini disampaikan bmkg dalam akun x resminya senin disebutkan bahwa gempa terjadi pada pukul wib pusat gempa berada di laut km barat laut pulau doi tulis bmkg titik gempa berada di koordinat lintang utara dan bujur timur bmkg mengatakan informasi gempa yang diberikan mengutamakan kecepatan sehingga dapat terjadi perubahan data gempa dirasakan dengan skala modified mercalli intensity mmi ii iii di tagulandang skala ii artinya getaran dirasakan oleh beberapa orang benda benda ringan yang digantung bergoyang sementara skala iii artinya getaran dirasakan nyata dalam rumah terasa getaran seakan akan ada truk berlalu
1,presiden prabowo subianto bertemu dengan gubernur banten terpilih andra soni di istana hari ini apa yang dibahas momen pertemuan diabadikan dan diunggah di akun instagram andra soni jumat andra mengenakan kemeja lengan panjang putih sementara prabowo memakai baju safari cokelat muda keduanya berfoto dengan latar belakang istana merdeka jakarta andra bersyukur mendapatkan kesempatan bertemu dengan prabowo alhamdulillah saya berkesempatan untuk bersilaturahmi dengan presiden republik indonesia bapak prabowo subianto tulis andra dalam keterangan di unggahan tersebut andra mendoakan prabowo selalu diberi kesehatan serta kesuksesan dalam memimpin negeri semoga beliau selalu diberikan kekuatan dan kesuksesan dalam memimpin negeri ini menuju indonesia maju ujarnya diketahui pasangan andra soni dimyati natakusumah mendapatkan suara terbanyak di pilgub banten hasil rekapitulasi kpu provinsi banten andra dimyati unggul atas pasangan airin rachmi diany ade sumardi di hasil pleno rekapitulasi penghitungan suara tingkat provinsi banten pasangan nomor urut airin ade mendapatkan jumlah total suara sedangkan pasangan nomor urut andra dimyati mendapatkan data rincian pasangan calon gubernur dan wakil gubernur nomor urut airin rachmi diany ade sumardi jumlah akhir nomor urut andra soni dimyati natakusumah akhir kata ketua kpu banten m ihsan dalam penetapan rekapitulasi tingkat provinsi di kpud banten sabtu adapun jumlah suara sah dan tidak sah di pilgub banten adalah jumlah suara tidak sah sebanyak suara jumlah suara sah dan tidak sah jumlah akhir hasil rekapitulasi ini ditandatangani semua anggota kpu provinsi banten termasuk ditandatangani oleh saksi dari pasangan calon nomor urut saksi pasangan calon yang menandatangani nomor urut ujarnya simak juga video kpu tetapkan andra soni dimyati natakusumah menang pilgub banten gambas video detik
2,jakarta kompas com wakil ketua mpr ri eddy soeparno mengingatkan pentignya mitigasi dan kesiapan manajemen krisis yang baik sebelum terjadi bencana besar hal ini disampaikan eddy merespons bencana yang membuat rt di jakarta dan kecamatan di bekasi terendam dengan kedalaman air yang bervariasi kita tidak bisa terus menerus hanya merespons saat bencana sudah terjadi perlu ada langkah mitigasi dan kesiapan manajemen krisis yang lebih baik agar dampaknya bisa diminimalkan ujar eddy selasa dikutip dari siaran pers politikus partai amanat nasional ini mengatakan bencana banjir ini merupakan bukti nyata dari krisis iklim yang semakin mengancam menurut dia pola banjir yang terus berulang perlu dihadapi dengan strategi yang lebih sistematis dalam memitigasi dampak perubahan iklim ini bukan pertama kalinya kita menghadapi banjir besar pola ini terus berulang setiap tahun dan kalau tidak ada kebijakan yang lebih serius maka ke depannya situasi bisa semakin buruk ujar eddy eddy pun mengingatkan para kepala daerah yang terpilih melalui pemilu harus memiliki kebijakan konkret dalam menangani krisis iklim dan bencana hidrometeorologi seperti banjir kepala daerah harus segera menyusun langkah strategis mulai dari perbaikan tata kelola air sistem drainase yang lebih baik hingga kesiapan tanggap darurat yang lebih cepat dan efektif jangan hanya bertindak ketika bencana suda

### Processing